## 1. Data

In [2]:
import os
os.environ["TF_ENABLE_ONEDNN_OPTS"] = "0"

import re
import numpy as np
import pandas as pd
import tensorflow as tf
import keras
from keras import layers
from keras_hub.layers import TransformerDecoder, TokenAndPositionEmbedding
import sentencepiece as spm
import kagglehub

SEED = 42
np.random.seed(SEED)
tf.random.set_seed(SEED)


I0000 00:00:1778139900.644706 1953001 pjrt_api.cc:95] PJRT_Api is set for device type gpu
I0000 00:00:1778139900.681114 1953001 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: SSE3 SSE4.1 SSE4.2 AVX AVX2 AVX512F AVX512_VNNI AVX512_BF16 AVX_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


In [3]:
# Can probably be commented out if you have no problems with GPU / CUDA!!!
import tensorflow as tf
print(tf.__version__)
print(tf.config.list_physical_devices("GPU"))
tf.config.optimizer.set_jit(False)
gpus = tf.config.list_physical_devices('GPU')
if gpus:
    tf.config.experimental.set_memory_growth(gpus[0], True)

2.22.0-dev0+selfbuilt
[PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]


In [4]:
path = kagglehub.dataset_download("rohitgr/wikitext")
wikitext2_path = os.path.join(path, "wikitext-2")

def load_lines(base_path, filename):
    with open(os.path.join(base_path, filename), "r", encoding="utf-8") as f:
        return f.readlines()


train_lines = load_lines(wikitext2_path, "wiki.train.tokens")
val_lines   = load_lines(wikitext2_path, "wiki.valid.tokens")
test_lines  = load_lines(wikitext2_path, "wiki.test.tokens")

## 2. Data Cleaning

In [5]:
def clean_line(text_line):
    text_line = text_line.strip()

    # Drop WikiText section titles; they are metadata, not natural sentences.
    if re.match(r"^=+\s*[^=]+?\s*=+$", text_line):
        return ""

    text_line = text_line.replace("<unk>", "")
    text_line = text_line.replace("@-@", "-")
    text_line = text_line.replace("@,@", ",")
    text_line = text_line.replace("@.@", ".")
    text_line = re.sub(r"\s+([,.;:!?])", r"\1", text_line)
    text_line = re.sub(r"([([{])\s+", r"\1", text_line)
    text_line = re.sub(r"\s+([)\]}])", r"\1", text_line)
    text_line = re.sub(r"\s+", " ", text_line)
    return text_line.strip()


train_lines_clean = [clean_line(text_line) for text_line in train_lines]
val_lines_clean   = [clean_line(text_line) for text_line in val_lines]
test_lines_clean  = [clean_line(text_line) for text_line in test_lines]

train_lines_clean = [line for line in train_lines_clean if len(line.split()) >= 4]
val_lines_clean   = [line for line in val_lines_clean if len(line.split()) >= 4]
test_lines_clean  = [line for line in test_lines_clean if len(line.split()) >= 4]


## 3. Subword Tokenizer

In [6]:
with open("wikitext_train.txt", "w", encoding="utf-8") as f:
    for line in train_lines_clean:
        f.write(line + "\n")

In [7]:
vocab_size_limit = 12000

if not os.path.exists("spm_wikitext.model"):
    for f in ["spm_wikitext.model", "spm_wikitext.vocab"]:
        if os.path.exists(f):
            os.remove(f)

    spm.SentencePieceTrainer.train(
        input="wikitext_train.txt",
        model_prefix="spm_wikitext",
        vocab_size=vocab_size_limit,
        model_type="bpe",
        character_coverage=1.0,
        pad_id=0,
        unk_id=1,
        bos_id=2,
        eos_id=3,
        input_sentence_size=500000,
    )

sp = spm.SentencePieceProcessor()
sp.load("spm_wikitext.model")


True

In [8]:
eos = sp.eos_id()

train_sequences = sp.encode(train_lines_clean, out_type=int)
val_sequences   = sp.encode(val_lines_clean, out_type=int)
test_sequences  = sp.encode(test_lines_clean, out_type=int)

In [9]:
train_sequences = [seq + [eos] for seq in train_sequences]
val_sequences   = [seq + [eos] for seq in val_sequences]
test_sequences  = [seq + [eos] for seq in test_sequences]

train_sequence = [tok for seq in train_sequences for tok in seq]
val_sequence   = [tok for seq in val_sequences for tok in seq]
test_sequence  = [tok for seq in test_sequences for tok in seq]

vocab_size = sp.get_piece_size()

print("Train tokens:", len(train_sequence))
print("Val tokens:", len(val_sequence))
print("Test tokens:", len(test_sequence))
print("Vocab size:", vocab_size)

Train tokens: 2326138
Val tokens: 230140
Test tokens: 256896
Vocab size: 12000


## 4. Dataset and Hyperparameters

In [10]:
# Hyperparameters
batch_size = 64
train_stride = 6
val_stride = 16
test_stride = 16
seq_len = 128
MAX_TRAIN_BATCHES = None  # Set to an integer, e.g. 4000, for a quicker training run

train_tokens = np.array(train_sequence, dtype=np.int32)
val_tokens   = np.array(val_sequence, dtype=np.int32)
test_tokens  = np.array(test_sequence, dtype=np.int32)


def make_dataset(tokens, seq_len, batch_size, shuffle=False, stride=64):
    tokens = np.asarray(tokens, dtype=np.int32)

    ds = tf.keras.utils.timeseries_dataset_from_array(
        data=tokens,
        targets=None,
        sequence_length=seq_len + 1,
        sequence_stride=stride,
        batch_size=batch_size,
        shuffle=False,
    )

    ds = ds.map(lambda window: (window[:, :-1], window[:, 1:]), num_parallel_calls=tf.data.AUTOTUNE)
    if shuffle:
        ds = ds.shuffle(2048, seed=SEED, reshuffle_each_iteration=True)
    return ds.prefetch(tf.data.AUTOTUNE)


train_ds = make_dataset(
    train_tokens,
    seq_len,
    batch_size,
    shuffle=True,
    stride=train_stride,
)

if MAX_TRAIN_BATCHES is not None:
    train_ds = train_ds.take(MAX_TRAIN_BATCHES)

val_ds = make_dataset(
    val_tokens,
    seq_len,
    batch_size,
    stride=val_stride,
)

test_ds = make_dataset(
    test_tokens,
    seq_len,
    batch_size,
    stride=test_stride,
)


I0000 00:00:1778139905.348395 1953001 gpu_device.cc:2042] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 9701 MB memory:  -> device: 0, name: NVIDIA GeForce GTX 1080 Ti, pci bus id: 0000:01:00.0, compute capability: 6.1


In [11]:
print("characters:", len(train_lines_clean))
print("tokens:", len(train_tokens))
print("steps:", tf.data.experimental.cardinality(train_ds).numpy())

characters: 22514
tokens: 2326138
steps: 6058


## 5. Model

In [12]:
embedding_dim = 256
embed_multiplier = 3
num_heads = 8
decoder_amount = 4
dropout_rate = 0.25

inputs = keras.Input(shape=(None,), dtype="int64", name="inputs")
x = TokenAndPositionEmbedding(
    vocabulary_size=vocab_size,
    sequence_length=seq_len,
    embedding_dim=embedding_dim,
    mask_zero=False,
)(inputs)

for _ in range(decoder_amount):
    x = TransformerDecoder(
        intermediate_dim=embedding_dim * embed_multiplier,
        num_heads=num_heads,
        dropout=dropout_rate,
    )(x)

x = layers.Dropout(dropout_rate)(x)
outputs = layers.Dense(vocab_size)(x)  # SparseCategoricalCrossentropy uses logits directly.

transformer_model = keras.Model(inputs, outputs, name="decoder_only_transformer")


In [13]:
transformer_model.summary()

Model: "decoder_only_transformer"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ inputs (InputLayer)             │ (None, None)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ token_and_position_embedding    │ (None, None, 256)      │     3,104,768 │
│ (TokenAndPositionEmbedding)     │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ transformer_decoder             │ (None, None, 256)      │       658,432 │
│ (TransformerDecoder)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ transformer_decoder_1           │ (None, None, 256)      │       658,432 │
│ (TransformerDecoder)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ transformer_decoder_2           │ (None, None, 256)      │       658,432 │
│ (TransformerDecoder)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ transformer_decoder_3           │ (None, None, 256)      │       658,432 │
│ (TransformerDecoder)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_4 (Dropout)             │ (None, None, 256)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, None, 12000)    │     3,084,000 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 8,822,496 (33.66 MB)

 Trainable params: 8,822,496 (33.66 MB)

 Non-trainable params: 0 (0.00 B)

## 6. Training

In [14]:
USE_EARLY_STOPPING = True


def make_callbacks(ckpt_path, use_early_stopping=USE_EARLY_STOPPING):
    callbacks = [
        keras.callbacks.ModelCheckpoint(
            ckpt_path,
            monitor="val_loss",
            mode="min",
            save_best_only=True,
            verbose=1,
        ),
        keras.callbacks.ReduceLROnPlateau(
            monitor="val_loss",
            mode="min",
            factor=0.5,
            patience=2,
            min_lr=1e-5,
            verbose=1,
        ),
    ]

    if use_early_stopping:
        callbacks.append(
            keras.callbacks.EarlyStopping(
                monitor="val_loss",
                mode="min",
                patience=4,
                min_delta=0.001,
                start_from_epoch=4,
                restore_best_weights=True,
                verbose=1,
            )
        )

    return callbacks


In [15]:
class Perplexity(keras.metrics.Metric):
    def __init__(self, name="perplexity", **kwargs):
        super().__init__(name=name, **kwargs)
        self.loss_tracker = keras.metrics.Mean()

    def update_state(self, y_true, y_pred, sample_weight=None):
        loss = keras.losses.sparse_categorical_crossentropy(
            y_true, y_pred, from_logits=True
        )

        self.loss_tracker.update_state(loss, sample_weight=sample_weight)

    def result(self):
        return tf.exp(self.loss_tracker.result())

    def reset_state(self):
        self.loss_tracker.reset_state()

In [ ]:
epochs = 24

# The saved model
if os.path.exists("best_model.keras"):
    transformer_model = keras.models.load_model(
    "best_model.keras",
    compile=False
)
    
transformer_model.compile(
    loss=keras.losses.SparseCategoricalCrossentropy(from_logits=True),
    metrics=[
        "accuracy",
        keras.metrics.SparseTopKCategoricalAccuracy(k=5, name="top_5_accuracy"),
        Perplexity(),
    ],
)

# THE ACTUAL TRAINABLE MODEL DO NOT REMOVE!!
#
# transformer_model.compile(
#     loss=keras.losses.SparseCategoricalCrossentropy(from_logits=True),
#     optimizer=keras.optimizers.AdamW(
#         learning_rate=2e-4,
#         weight_decay=2e-4,
#         clipnorm=1.0,
#     ),
#     metrics=[
#         "accuracy",
#         keras.metrics.SparseTopKCategoricalAccuracy(k=5, name="top_5_accuracy"),
#         Perplexity(),
#     ],
#     jit_compile=False,
# )

# transformer_history = transformer_model.fit(
#     train_ds,
#     validation_data=val_ds,
#     epochs=epochs,
#     callbacks=make_callbacks("best_model.keras"),
# )


W0000 00:00:1778139907.531601 1953001 cpu_allocator_impl.cc:82] Allocation of 12288000 exceeds 10% of free system memory.
W0000 00:00:1778139907.651025 1953001 cpu_allocator_impl.cc:82] Allocation of 12288000 exceeds 10% of free system memory.


In [17]:
# for loss in transformer_history.history["val_loss"]:
#     print(np.exp(loss))

In [18]:
test_results = transformer_model.evaluate(test_ds, verbose=1)

for name, value in zip(transformer_model.metrics_names, test_results):
    print(f"{name}: {value:.4f}")

I0000 00:00:1778139908.199069 1954846 service.cc:178] XLA service 0x78604803cb00 initialized for platform CUDA (this does not guarantee that XLA will be used). Devices:
I0000 00:00:1778139908.199111 1954846 service.cc:194]   StreamExecutor [0]: NVIDIA GeForce GTX 1080 Ti, Compute Capability 6.1 (Driver: 13.0.0[582.28.0]; Runtime: 12.3.0; Toolkit: 12.3.0; DNN: 8.9.7)
I0000 00:00:1778139908.256650 1954846 dump_mlir_util.cc:269] disabling MLIR crash reproducer, set env var `MLIR_CRASH_REPRODUCER_DIRECTORY` to enable.
W0000 00:00:1778139908.371565 1954846 assert_op.cc:39] Ignoring Assert operator compile_loss/sparse_categorical_crossentropy/SparseSoftmaxCrossEntropyWithLogits/assert_equal_1/Assert/Assert
W0000 00:00:1778139908.371749 1954846 assert_op.cc:39] Ignoring Assert operator SparseSoftmaxCrossEntropyWithLogits/assert_equal_1/Assert/Assert
I0000 00:00:1778139908.433000 1954846 cuda_dnn.cc:461] Loaded cuDNN version 8907


  5/251 ━━━━━━━━━━━━━━━━━━━━ 7s 29ms/step - accuracy: 0.2527 - loss: 4.5041 - perplexity: 90.6982 - top_5_accuracy: 0.4697

I0000 00:00:1778139909.090467 1954846 device_compiler.h:208] Compiled cluster using XLA!  This line is logged at most once for the lifetime of the process.


249/251 ━━━━━━━━━━━━━━━━━━━━ 0s 26ms/step - accuracy: 0.2648 - loss: 4.4629 - perplexity: 86.8753 - top_5_accuracy: 0.4666

W0000 00:00:1778139915.761286 1954846 assert_op.cc:39] Ignoring Assert operator compile_loss/sparse_categorical_crossentropy/SparseSoftmaxCrossEntropyWithLogits/assert_equal_1/Assert/Assert
W0000 00:00:1778139915.761339 1954846 assert_op.cc:39] Ignoring Assert operator SparseSoftmaxCrossEntropyWithLogits/assert_equal_1/Assert/Assert


251/251 ━━━━━━━━━━━━━━━━━━━━ 9s 29ms/step - accuracy: 0.2630 - loss: 4.5256 - perplexity: 92.3484 - top_5_accuracy: 0.4631
loss: 4.5256
compile_metrics: 0.2630


## 7. Sampling

In [19]:
def sample_probs(logits, temperature=1.0):
    temperature = max(float(temperature), 1e-6)
    logits = np.asarray(logits).astype("float64") / temperature
    logits = logits - np.max(logits)
    exp_logits = np.exp(logits)
    return exp_logits / np.sum(exp_logits)


def top_p_filter(probs, top_p=0.9, min_tokens_to_keep=3):
    sorted_idx = np.argsort(probs)[::-1]
    sorted_probs = probs[sorted_idx]
    cumulative = np.cumsum(sorted_probs)

    keep = cumulative <= top_p
    keep[:min_tokens_to_keep] = True
    filtered_idx = sorted_idx[keep]
    filtered_probs = probs[filtered_idx]
    filtered_probs = filtered_probs / filtered_probs.sum()
    return filtered_idx, filtered_probs


In [20]:
BAD_PIECES = {
    "<pad>", "<unk>", "<s>",
    "<", ">", "▁<", "▁>",
    "__&", "__/",
    "@-@", "▁@-@", " "
}

BAD_IDS = {
    sp.pad_id(),
    sp.unk_id(),
    sp.bos_id(),
}

for piece in BAD_PIECES:
    piece_id = sp.piece_to_id(piece)
    if piece_id != sp.unk_id():
        BAD_IDS.add(piece_id)


def encode_prompt(text):
    seq = sp.encode(text, out_type=int)
    seq = seq[-seq_len:]
    return np.array([seq], dtype=np.int32)


def repetition_adjusted_logits(logits, generated_ids, penalty=1.2, recent_window=80, hard_window=6):
    logits = np.array(logits, dtype="float64", copy=True)

    for bad_id in BAD_IDS:
        if 0 <= bad_id < len(logits):
            logits[bad_id] = -1e9

    recent_ids = generated_ids[-recent_window:]
    hard_block_ids = set(generated_ids[-hard_window:])

    for token_id in set(recent_ids):
        if 0 <= token_id < len(logits):
            if token_id in hard_block_ids:
                logits[token_id] = -1e9
            elif logits[token_id] > 0:
                logits[token_id] /= penalty
            else:
                logits[token_id] *= penalty

    return logits


def predict_top_n(text, model, top_n=5, temperature=0.8, filter_bad=True):
    seq = encode_prompt(text)
    logits = model.predict(seq, verbose=0)[0, -1]

    if filter_bad:
        logits = repetition_adjusted_logits(logits, sp.encode(text, out_type=int), penalty=1.05)

    probs = sample_probs(logits, temperature)
    sorted_indices = np.argsort(probs)[::-1]

    results = []
    for token_id in sorted_indices:
        piece = sp.id_to_piece(int(token_id))
        decoded = sp.decode([int(token_id)])
        if not decoded.strip() and piece != "▁":
            continue

        results.append((decoded, round(float(probs[token_id]) * 100, 2)))
        if len(results) == top_n:
            break

    return results


def sample_next_token_id(generated_ids, model, temperature=0.8, top_k=50, top_p=0.9):
    seq = np.array([generated_ids[-seq_len:]], dtype=np.int32)
    logits = model.predict(seq, verbose=0)[0, -1]
    logits = repetition_adjusted_logits(logits, generated_ids)

    probs = sample_probs(logits, temperature)
    top_indices = np.argsort(probs)[::-1][:top_k]
    top_probs = probs[top_indices]
    top_probs = top_probs / top_probs.sum()

    if top_p is not None:
        kept_local_idx, kept_probs = top_p_filter(top_probs, top_p=top_p)
        top_indices = top_indices[kept_local_idx]
        top_probs = kept_probs

    return int(np.random.choice(top_indices, p=top_probs))


def generate_text(seed_text, model, num_tokens=60, temperature=0.75, top_k=60, top_p=0.9):
    generated_ids = sp.encode(seed_text, out_type=int)

    for _ in range(num_tokens):
        next_id = sample_next_token_id(
            generated_ids,
            model,
            temperature=temperature,
            top_k=top_k,
            top_p=top_p,
        )

        if next_id == sp.eos_id():
            break

        generated_ids.append(next_id)

    text = sp.decode(generated_ids)
    text = re.sub(r"\s+([,.;:!?])", r"\1", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text


In [24]:
# Tuneable settings for making predictions
number_of_tokens = 40
settings = [
    ("safe", 0.35, 15, 0.8),
    ("balanced", 0.6, 40, 0.9),
    ("creative", 0.8, 80, 0.95),
]

seeds = seeds = [
    "The film received generally",
    "The album was released in",
    "The war ended in",
    "The character is described as",
    "in her early years, she was",
    "the city is located in",
    "the population of the town was",
    "according to the report",
]

print("\n=== Transformer: Top-5 seuraavaa sanaa ===")
for i in range(len(seeds)):
    print("------------------")
    for piece, pct in predict_top_n(seeds[i], transformer_model, top_n=5):
        print(f"  {piece:20s} {pct:.2f}%")

print("\n=== Transformer generoitu teksti ===")
for name, temp, top_k, top_p in settings:
    print("------------------")
    for seed in seeds:
        print()
        print(generate_text(
            seed,
            transformer_model,
            num_tokens=number_of_tokens,
            temperature=temp,
            top_k=top_k,
            top_p=top_p,
        ))



=== Transformer: Top-5 seuraavaa sanaa ===
------------------
  mixed                45.18%
  positive             42.52%
  favorable            11.09%
  negative             0.36%
  nominations          0.11%
------------------
  the                  32.49%
  June                 5.43%
  Japan                4.49%
  November             4.16%
  September            3.51%
------------------
  the                  12.44%
  June                 7.44%
  a                    5.84%
  November             5.37%
  October              4.27%
------------------
  a                    37.79%
  "                    34.66%
  the                  8.69%
  an                   6.46%
  one                  4.08%
------------------
  the                  14.22%
  a                    12.67%
  named                8.18%
  born                 4.93%
  not                  2.88%
------------------
  a                    27.29%
  Varanasi             14.29%
  an                   3.59%
  its              